# Setup

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import os
import json
import time
import utils
# Set device
device = torch.device("cuda")
print(f"Using device: {device}")

In [ ]:
# Define the path to the dataset containing all images
DATA_DIR = "./maps"
# Define batch size
BATCH_SIZE = 1
# Define CPU threads to load prepare images while training
NUM_WORKERS = 2

# Load transforms 
transforms = utils.get_transforms()

# Load 2 intances, one with augmentation for training and other without for validation and test


dataset = utils.MapsDataset(root_dir=DATA_DIR, transform=transforms, augment=False)

# Ensure same indices for both
total_size = len(dataset)
indices = torch.randperm(total_size, generator=torch.Generator().manual_seed(42)).tolist()

# Split limits
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)

# Split

test_indices = indices[train_size + val_size :]

# Create the subset

test_dataset = torch.utils.data.Subset(dataset, test_indices)

# Init dataloader
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Total images found: {total_size}")

print(f"Allocated for Testing (Deterministic): {len(test_dataset)}")

# Load the model

In [ ]:
GEN_PATH = "trained_models/arch_1/model_1_lambda_100_long/epoch_200_G_lambda_100.pth"
# Init the generator architecture
generator = utils.GeneratorUNet(in_channels=3, out_channels=3).to(device)

# Load only the weights of the generator to test
generator.load_state_dict(torch.load(GEN_PATH, map_location=device))
generator.eval()
print("Generator loaded and ready for test.")

In [ ]:
# Load history and plot training results
with open("trained_models/arch_1/model_1_lambda_100_long/history_lambda_100.json", 'r') as f:
    history_from_json = json.load(f)


utils.plot_training_history(history_from_json, earlystopping=False)

## Compute metrics for the given  model

In [ ]:
utils.evaluate_test_metrics(generator, test_loader, device)

In [ ]:
# ==========================================
# MODEL EVALUATION & VISUALIZATION
# ==========================================
n = 50 # Load all the images in the test set
indexes = range(n) 

# Call our function on the test set
utils.model_benchmark(
    generator=generator, 
    dataset=test_dataset, 
    device=device, 
    num_samples=len(indexes), 
    indexes=indexes
)